# kwabena: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [6_mcp/community_contributions/kwabena/evaluator_server.py](../../../../../../6_mcp/community_contributions/kwabena/evaluator_server.py)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 2. Evaluate, give feedback, and bound revision

A worker produces a draft and an evaluator judges it against criteria. Feedback becomes input for another attempt. The evaluator here is a small deterministic teaching rule, not a general quality checker.

**Predict:** Is the first draft acceptable? How many attempts will be allowed?

In [ ]:
def evaluate(text):
    return {"accepted": "example:" in text.lower(), "feedback": "Include an example."}
attempts = []
for attempt in range(2):
    draft = "A list stores ordered values." if attempt == 0 else "A list stores ordered values. Example: [1, 2]"
    decision = evaluate(draft)
    attempts.append(decision["accepted"])
    if decision["accepted"]:
        break
print(attempts, draft)
assert attempts == [False, True]

**Check:** The first is rejected and the second accepted. Unlike the Lab 3 one-retry source, this teaching example evaluates the revision too.

**Change one thing:** Make the second draft fail too. Add an explicit message when the finite retry limit is reached.

## 3. MCP tools, resources, and server lifetime

A tool performs an operation; a resource supplies readable context. A client discovers tool descriptions and calls them while the server is connected. This fixture is not a protocol transport.

**Predict:** Is listing tool names the same as executing them?

In [ ]:
from contextlib import contextmanager
calls = []
@contextmanager
def practice_server():
    tools = {"add": lambda a, b: a + b}
    try:
        calls.append("connected")
        yield tools
    finally:
        calls.append("closed")
with practice_server() as tools:
    print("Discovered:", list(tools))
    result = tools["add"](12, 30)
    print("Called add:", result)
assert result == 42 and calls == ["connected", "closed"]

**Check:** No. Discovery returns descriptions. A real stdio subprocess also has a working directory, environment, and protocol stream.

**Change one thing:** Add a read-only dictionary resource and a tool returning a fixed date. Explain how discovery differs from invocation.

## 4. Register a real local FastMCP tool without starting a server

The decorator registers a Python function with FastMCP. Tool listing here happens directly in process; it neither launches a subprocess nor calls a hosted model.

**Predict:** Which tool name will be discovered?

In [ ]:
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.server import Settings
from unittest.mock import patch
# FastMCP's settings default to reading a working-directory .env.
# Disable only that file lookup while constructing this isolated fixture.
with patch.dict(Settings.model_config, {"env_file": None}):
    server = FastMCP("offline-practice")
@server.tool()
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b
descriptions = await server.list_tools()
print([item.name for item in descriptions])
assert "add" in [item.name for item in descriptions]

**Check:** add. Direct registration is verified locally; client/server transport remains a separate integration.

**Change one thing:** Register a second harmless function. To expose stdio later, use a separate script with server.run(transport="stdio"); stdout then carries protocol messages.

## 5. Simulated accounts change state

Buying shares updates cash, holdings, and a transaction record. Practice against a dictionary so you do not import account modules that create or alter databases.

**Predict:** How much cash remains after buying three fictional shares at 100 each?

In [ ]:
account = {"cash": 1000.0, "holdings": {}, "transactions": []}
def buy(symbol, quantity, price):
    cost = quantity * price
    if quantity <= 0 or price <= 0 or cost > account["cash"]:
        raise ValueError("Invalid or unaffordable order")
    account["cash"] -= cost
    account["holdings"][symbol] = account["holdings"].get(symbol, 0) + quantity
    account["transactions"].append({"symbol": symbol, "quantity": quantity, "price": price})
buy("DEMO", 3, 100)
print(account)
assert account["cash"] == 700 and account["holdings"]["DEMO"] == 3

**Check:** 700. These are fictional values and this code places no real order.

**Change one thing:** Try an unaffordable order and confirm the balance is unchanged. Explain why reset is a data-changing action.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.